# Are the foundation models actually *significantly* better? — TSFM significance testing on TSBench-Forge

This notebook puts **up to 10 pretrained time-series foundation models** (Chronos-2, TimesFM-2.5,
Moirai-2, Toto-2, TiRex, FlowState, Sundial, TabPFN-TS, Chronos-Bolt, TimesFM-2.0) on the
**live TSBench-Forge benchmark** (real, freshly-scraped public series) and asks not just *who ranks
first* but **whether the gaps are real** — via paired **Wilcoxon signed-rank** and **paired t-tests**,
a **Friedman** omnibus, and multiple-comparison correction.

**Hypothesis.** On our real-data challenge set, the strongest TSFMs beat the seasonal-naive baseline
by a margin that survives a paired significance test at α=0.05, and the *ordering among the top TSFMs*
is mostly **not** individually significant (they cluster).

**Why paired tests.** Every model forecasts the *same* challenges, so per-challenge errors are paired.
Wilcoxon is the robust default for heavy-tailed forecast errors; the paired t-test is the parametric
companion; win-rate is the distribution-free effect size.

> ⚠️ **Breadth caveat, read first.** The benchmark forecasts *real* data and the fresh catalog is
> still accumulating. Challenges are drawn from a small number of source series right now, so they are
> **not fully independent** — the tests below treat each challenge as a unit, which *overstates* power
> when the source count is low. The notebook prints the source count and the primary ranking remains
> the per-source-relative shifted-gmean leaderboard; treat the pairwise p-values as corroboration. As
> the scraper's cron accumulates more sources/history, re-run and the breadth caveat weakens.

**Runs anywhere.** All model libraries are imported lazily and any model that fails to load is skipped,
so the notebook runs on CPU (classical panel only) for a plumbing check and on a **modest GPU (L40 / RTX 4090)**
for the full roster. See the last section for the one-command Lium runner.

## 1 · Config

In [ ]:
import os, sys, json
import numpy as np
sys.path.insert(0, os.path.abspath("../src"))  # importable core

import tsfm_comparison as tc
import model_comparison as mc

# --- knobs ---
ROSTER      = tc.DEFAULT_ROSTER          # the 10-model roster (short names)
DEVICE      = "cuda" if os.environ.get("FORCE_CPU") != "1" else "cpu"
MOTIF_LEN   = 304                        # CONTEXT_LEN(256) + HORIZON(48); series must be >= this
N_CHALLENGES= 256
SEED        = "tsfm-significance-v1"

# --- locate scraped parquet: local repo, else committed test fixture (offline) ---
CANDIDATES = ["../src/sources/data", "../tests/fixtures/sources_data"]
DATA_DIR = next((p for p in CANDIDATES if os.path.isdir(p) and os.listdir(p)), CANDIDATES[0])
CATALOG  = "../src/sources/sources.yaml"
print("device   :", DEVICE)
print("data_dir :", DATA_DIR)
print("roster   :", list(ROSTER))

## 2 · Build the challenge set (deterministic, commit-reveal seeded)

In [ ]:
from scraped_source import ScrapedLiveSource
from ingest import FreshBuffer
from collections import Counter

# Eligible universe: series with >= MOTIF_LEN points (context+horizon). This is the
# forecasting bar; it is MUCH higher than the ~30-point bar the PCA feature-space
# notebook uses (which is why that notebook sees ~10x more series). The loader now
# concatenates each source's accumulated daily parquet, so this universe grows with
# the cron. A single challenge set then SAMPLES a subset of this universe.
_src = ScrapedLiveSource(CATALOG, DATA_DIR, min_series_length=MOTIF_LEN)
n_eligible = len(_src._catalog())

challenges = tc.build_challenges(
    DATA_DIR, catalog=CATALOG, motif_len=MOTIF_LEN, n_challenges=N_CHALLENGES, seed=SEED,
)
srcs = [ (getattr(ch,'meta',None) or {}).get('source_id') for ch in challenges ]
doms = [ (getattr(ch,'meta',None) or {}).get('domain') for ch in challenges ]
print(f"eligible universe: {n_eligible} series (>= {MOTIF_LEN} pts) — grows as the cron accumulates history")
print(f"this run: {len(challenges)} challenges sampling {len(set(srcs))} of those series | domains={dict(Counter(doms))}")

## 3 · Load the models

Each adapter is **probed with a tiny forecast** at load time, so "loaded" means "actually runnable
on this box." Missing extras → the model is skipped and listed below; the comparison proceeds on
whatever loaded, always including the classical reference panel (seasonal-naive, drift, AR(1), EWMA,
a strong classical ensemble, and a context-parrot lower bound).

In [ ]:
import pandas as pd
models, load_report = tc.load_models(ROSTER, device=DEVICE)
disp = pd.DataFrame(load_report)
loaded_tsfms = [r['name'] for r in load_report if r['loaded']]
print(f"TSFMs loaded: {loaded_tsfms or '(none — running classical-only plumbing check)'}")
print(f"total models scored (incl. classical rungs): {len(models)}")
disp

## 4 · Primary ranking — seasonal-naive-relative leaderboard (MASE / WQL / CRPS)

In [ ]:
from evaluate import leaderboard
board = leaderboard(models, challenges)
cols = ["rank","model","mase_rel","crps_rel","mase","wql","crps","coverage_err"]
bdf = pd.DataFrame([{k: r.get(k) for k in cols} for r in board]).set_index("rank")
bdf.round(3)   # (.style gradients render in Jupyter; plain here for portability)

## 5 · Significance tests

* **Friedman omnibus** — is *any* model different? If it can't reject, stop reading the pairwise table.
* **Vs seasonal-naive** — each model's per-challenge CRPS paired against the naive baseline, one-sided
  (is it *better*?), **Holm**-corrected across the roster.
* **Pairwise** — all-pairs Wilcoxon, **Benjamini-Hochberg** FDR across the family.

In [ ]:
scores = mc.score_models(models, challenges)
note = mc.source_clustered_note(scores)
print(note, "\n")
for metric in ("crps","mase"):
    fr = mc.friedman_omnibus(scores, metric=metric)
    print(f"Friedman [{metric}]:  chi2={fr['statistic']:.1f}  df={fr['df']:.0f}  p={fr['p_value']:.3g}"
          f"  -> {'models differ' if fr['p_value']<0.05 else 'NOT significant'}")

In [ ]:
# --- vs seasonal-naive (CRPS), Holm-corrected ---
vb = mc.compare_to_baseline(scores, metric="crps")
vdf = pd.DataFrame(vb)[["model","median_diff","win_rate","wilcoxon_p","wilcoxon_p_holm","t_p_holm","significant"]]
print("Negative median_diff = better than seasonal-naive.  significant = Holm-adjusted Wilcoxon p < 0.05")
vdf.round(4)

In [ ]:
# --- pairwise BH-adjusted Wilcoxon p-values (CRPS) ---
pw = mc.pairwise_significance(scores, metric="crps")
P = pd.DataFrame(pw["p"], index=pw["names"], columns=pw["names"])
print("Rows/cols ordered best->worst by weighted-mean CRPS. Cell = BH-adjusted p that the pair differs.")
P.round(4)

## 6 · Figures

In [ ]:
import matplotlib.pyplot as plt

def leaderboard_bar(board, ax, metric="crps_rel"):
    rows = [r for r in board if r.get(metric) is not None]
    rows = sorted(rows, key=lambda r: r[metric])
    names = [r["model"] for r in rows]; vals = [r[metric] for r in rows]
    colors = ["#2a9d8f" if v < 1 else "#e76f51" for v in vals]
    ax.barh(names, vals, color=colors)
    ax.axvline(1.0, color="k", lw=1, ls="--", label="seasonal-naive")
    ax.invert_yaxis(); ax.set_xlabel(metric + "  (<1 beats seasonal-naive)")
    ax.set_title("Leaderboard — seasonal-naive-relative CRPS"); ax.legend(loc="lower right")

def sig_heatmap(pw, ax, alpha=0.05):
    names, P = pw["names"], np.array(pw["p"])
    im = ax.imshow(P, cmap="viridis_r", vmin=0, vmax=alpha*2)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=90, fontsize=7)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7)
    for i in range(len(names)):
        for j in range(len(names)):
            if i!=j and P[i,j] < alpha:
                ax.text(j, i, "*", ha="center", va="center", color="w", fontsize=10)
    ax.set_title(f"Pairwise CRPS: BH-adj p  (* = sig @ {alpha})")
    plt.colorbar(im, ax=ax, fraction=0.046)

def vs_baseline_bar(vb, ax):
    vb = sorted(vb, key=lambda r: r["median_diff"])
    names = [r["model"] for r in vb]; med = [r["median_diff"] for r in vb]
    colors = ["#2a9d8f" if r["significant"] else "#adb5bd" for r in vb]
    ax.barh(names, med, color=colors); ax.axvline(0, color="k", lw=1)
    ax.invert_yaxis(); ax.set_xlabel("median per-challenge CRPS diff vs naive  (<0 better)")
    ax.set_title("Improvement over seasonal-naive\n(green = Holm-significant)")

def winrate_heatmap(pw, ax):
    names, W = pw["names"], np.array(pw["win"])
    im = ax.imshow(W, cmap="RdBu", vmin=0, vmax=1)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=90, fontsize=7)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7)
    ax.set_title("Win-rate  W[i,j]=P(row i beats col j)")
    plt.colorbar(im, ax=ax, fraction=0.046)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
leaderboard_bar(board, axes[0,0])
vs_baseline_bar(vb, axes[0,1])
sig_heatmap(pw, axes[1,0])
winrate_heatmap(pw, axes[1,1])
fig.suptitle("TSFM significance on TSBench-Forge live data", fontsize=14)
fig.tight_layout()
os.makedirs("figures", exist_ok=True)
fig.savefig("figures/tsfm_significance.png", dpi=120, bbox_inches="tight")
plt.show()

## 7 · Persist results

In [ ]:
out = tc.run_comparison(DATA_DIR, catalog=CATALOG, roster=ROSTER, device=DEVICE, motif_len=MOTIF_LEN,
                        n_challenges=N_CHALLENGES, seed=SEED, out_dir="results")
print("wrote results/results.json")
print("loaded:", [r["name"] for r in out["load_report"] if r["loaded"]])
print("note  :", out["note"])

## 8 · Running the full roster on a Lium GPU

The classical panel runs on CPU; the TSFMs need a GPU. From the repo root:

```bash
# LIUM_API_KEY must be in your local .env (it is NEVER uploaded to the pod).
python scripts/run_tsfm_comparison_lium.py --gpu L40 --group A
```

`--group A` = the coexisting models `{chronos2, chronos-bolt, timesfm25, toto2, tirex, moirai2, flowstate, tabpfn-ts}`.
`--group B` = `{sundial}` in its own env (it pins `transformers==4.40.1`). The runner rsyncs the repo
(excluding `.env`, `.git`, and the heavy `data/`), stages a slice of the scraped parquet, installs the
group's extras, runs `tsfm_comparison.run_comparison` on the GPU, and pulls `results/results.json` +
`figures/` back. It **always tears the pod down** (try/finally), even on failure.

Merge group results by concatenating each `results.json`'s `load_report`/`leaderboard` — the challenge
set is identical across groups because `(SEED, MOTIF_LEN, N_CHALLENGES, DATA_DIR)` is fixed.